# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Across FlyRank's indexed content, which pages are underperforming the click-through rate their search position should earn — and what specific action should a reviewer take on each one first?

**Decision this supports:** ordering the SEO team's review queue. It decides which pages a human looks at first, not whether any specific fix will change a ranking, and nothing here auto-publishes a change.

In [ ]:
MONTH = "2026-03"
PRIOR_MONTH = "2026-02"          # used for the time-aware validation in Section 3
ELIGIBILITY_MIN_IMPRESSIONS = 100  # below this, a CTR estimate is too noisy to trust
SCORE_THRESHOLD = 0.0005          # 0.05 percentage points

print(f"Scoring month: {MONTH}")
print(f"Eligibility gate: impressions >= {ELIGIBILITY_MIN_IMPRESSIONS}")
print(f"Flag threshold: predicted_ctr - actual_ctr > {SCORE_THRESHOLD}")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Built on `FlyRank/internship-warehouse` (Hugging Face, gated). Core table: `fact_content_daily_performance`, March 2026 partition, joined against `dim_content` for content-level features.

9,841,378 total rows in the March partition; 7,700,646 (~78%) survive the join-integrity filter against GSC and GA4 client data. Aggregated to one row per content item, then filtered to the eligibility gate (≥100 monthly impressions), producing the analysis set used below.

`fact_content_query_90d` was intentionally excluded — it's the warehouse's sealed, held-out table and was not touched during development. No client names, domains, URLs, private queries, credentials, or raw exports appear anywhere in this notebook or its outputs.

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{userdata.get("HF_TOKEN")}'
);
""")

MARCH_PATH = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_query = f"""
WITH agg AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{MARCH_PATH}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= {ELIGIBILITY_MIN_IMPRESSIONS}
)
SELECT
    a.client_hash_id, a.content_hash_id,
    a.avg_position, a.impressions,
    d.main_intent, d.search_volume, d.category_count,
    a.clicks * 1.0 / NULLIF(a.impressions, 0) AS actual_ctr
FROM agg a
JOIN read_parquet('{CONTENT_PATH}') d ON a.content_hash_id = d.content_hash_id
"""
df = con.sql(feature_query).df().dropna(subset=["actual_ctr", "avg_position", "main_intent",
                                                 "search_volume", "category_count", "impressions"])
numeric_cols = ["avg_position", "impressions", "search_volume", "category_count"]
df[numeric_cols] = df[numeric_cols].astype("float64")
print(f"Eligible analysis set: {df.shape[0]} rows, {df.shape[1]} columns")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Features:** impression-weighted average position, impressions, search volume, category count (numeric); main search intent (categorical, one-hot encoded). **Label:** actual CTR (clicks ÷ impressions).

**Baseline:** expected CTR read off a five-band position lookup table (1–3, 4–6, 7–10, 11–20, 21+). The CTR-vs-position relationship was confirmed to hold across all five bands before any model was built.

**Model:** Random Forest (300 trees, max depth 8) — chosen over a single decision tree for stability, since roughly a third of eligible items are tied at exactly zero clicks.

**Validation design — two independent leakage checks, both run below:**
1. **Client-grouped split** — rows from the same client never appear in both train and test. A naive random split gives a falsely optimistic MAE, because the model partly memorizes a client's typical CTR level rather than the general position-CTR relationship.
2. **Time-aware split** — trained on February, tested on March, to confirm the relationship holds across time and not just across clients.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

numeric_features = ["avg_position", "impressions", "search_volume", "category_count"]
categorical_features = ["main_intent"]

def make_model():
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("pre", pre),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])

X = df[numeric_features + categorical_features]
y = df["actual_ctr"]

# --- Baseline: position-band bucket average ---
def position_band(pos):
    if pos <= 3: return "1-3"
    elif pos <= 6: return "4-6"
    elif pos <= 10: return "7-10"
    elif pos <= 20: return "11-20"
    else: return "21+"

df["_band"] = df["avg_position"].apply(position_band)
band_avg_ctr = df.groupby("_band")["actual_ctr"].mean()
baseline_pred = df["_band"].map(band_avg_ctr)
baseline_mae = mean_absolute_error(y, baseline_pred)

# --- Leakage check: naive random split vs client-grouped split ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
naive_model = make_model()
naive_model.fit(X_train_r, y_train_r)
naive_mae = mean_absolute_error(y_test_r, naive_model.predict(X_test_r))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_hash_id"]))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

grouped_model = make_model()
grouped_model.fit(X_train_g, y_train_g)
grouped_mae = mean_absolute_error(y_test_g, grouped_model.predict(X_test_g))

print(f"Baseline MAE: {baseline_mae:.5f}")
print(f"Naive random-split MAE: {naive_mae:.5f}  (falsely optimistic — confirms the leak)")
print(f"Client-grouped-split MAE: {grouped_mae:.5f}")

In [ ]:
# --- Time-aware validation: train on February, test on March ---
FEB_PATH = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={PRIOR_MONTH}/*.parquet"

feb_query = f"""
WITH agg AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{FEB_PATH}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= {ELIGIBILITY_MIN_IMPRESSIONS}
)
SELECT
    a.client_hash_id, a.content_hash_id,
    a.avg_position, a.impressions,
    d.main_intent, d.search_volume, d.category_count,
    a.clicks * 1.0 / NULLIF(a.impressions, 0) AS actual_ctr
FROM agg a
JOIN read_parquet('{CONTENT_PATH}') d ON a.content_hash_id = d.content_hash_id
"""
df_feb = con.sql(feb_query).df().dropna(subset=["actual_ctr", "avg_position", "main_intent",
                                                  "search_volume", "category_count", "impressions"])
df_feb[numeric_cols] = df_feb[numeric_cols].astype("float64")

time_aware_model = make_model()
time_aware_model.fit(df_feb[numeric_features + categorical_features], df_feb["actual_ctr"])
time_aware_mae = mean_absolute_error(y, time_aware_model.predict(X))

# within-March MAE, same random split as the leakage check above, for direct comparison
within_march_mae = naive_mae

print(f"Time-aware (train Feb, test March) MAE: {time_aware_mae:.5f}")
print(f"Within-March (random split) MAE for comparison: {within_march_mae:.5f}")
print(f"Gap: {abs(time_aware_mae - within_march_mae):.5f}")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| Model | Split | MAE |
|---|---|---|
| Position-band baseline | — (in-sample) | 0.00250 |
| Random Forest | Client-grouped (held-out) | 0.00268 (−7.0% vs. baseline) |
| Random Forest | Within-March (random, held-out) | 0.00243 |
| Random Forest | Time-aware (Feb→March, held-out) | 0.00243 (0.0% gap) |

**Honest read:** the baseline was scored in-sample (fit and evaluated on the same full dataset), while every Random Forest number above is properly held out — so the −7.0% comparison is optimistic *for* the baseline, not a fair fight. Even so, on the strictest available test (client-grouped, held-out), this model does not clearly beat a simple five-bucket lookup table. It does outperform on the time-aware split, and shows no generalization gap across time. The model's practical value here is the structure it adds (a continuous score, reason codes, archetype-based actions), not a proven accuracy edge over the baseline.

In [ ]:
from sklearn.inspection import permutation_importance

deployment_model = make_model()
deployment_model.fit(X, y)

perm = permutation_importance(deployment_model, X_test_g, y_test_g,
                               n_repeats=10, random_state=42, n_jobs=-1)
importance_dict = dict(zip(numeric_features + categorical_features, perm.importances_mean.round(4)))

results_table = {
    "baseline_mae": round(baseline_mae, 5),
    "grouped_split_mae": round(grouped_mae, 5),
    "within_march_mae": round(within_march_mae, 5),
    "time_aware_mae": round(time_aware_mae, 5),
    "improvement_over_baseline_pct": round((baseline_mae - grouped_mae) / baseline_mae * 100, 1),
    "permutation_importance": importance_dict,
}
print(results_table)

## 5. Limitations

*What this work cannot claim.*

- **Cross-sectional, single month.** No intervention was run — this supports "these pages look worth reviewing, because predicted CTR exceeds actual," never "fixing the title will raise the CTR."
- **Staleness/refresh timing could not be tested independently.** A separate check (`w04_signal_audit.ipynb`) found the eligibility gate itself correlated with the staleness variable, leaving only 69, 32, and 0 items across three older bands against 88,985 in the freshest — too little data to conclude either way. Reported as a negative result, not hidden.
- **The eligibility gate is named wherever it's used** — the 99,197-item analysis set excludes anything under 100 monthly impressions, and that exclusion isn't neutral for low-traffic content.
- **Worst errors under-predict outlier-high-CTR items** — the model is more conservative than reality for the small number of pages that dramatically beat their position's expected CTR.

In [ ]:
# From w04_signal_audit.ipynb — staleness/refresh check, carried over for reference
staleness_bands_from_signal_audit = {
    "older_band_1": 69, "older_band_2": 32, "older_band_3": 0, "freshest_band": 88985
}
print("Staleness check sample sizes by band (from w04_signal_audit.ipynb):")
print(staleness_bands_from_signal_audit)
print("Too few older items to test the staleness flag independently of the eligibility gate.")

zero_click_share = (df["actual_ctr"] == 0).mean() * 100
print(f"\nShare of the eligible set tied at exactly zero clicks: {zero_click_share:.1f}%")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Every flagged item is routed to one action based on its search position, because the right next step depends on whether the page already has visibility it isn't converting, or hasn't earned visibility yet:

| Archetype | Position | Action |
|---|---|---|
| High-rank, ignored | 1–6 | Review snippet & title |
| Mid-rank, ignored | 7–10 | Review snippet/title + content-relevance check |
| Low-rank, ignored | 11+ | Improve ranking signals |

Nothing here is auto-published — every flagged item passes through human review before any change goes live.

In [ ]:
df["predicted_ctr"] = deployment_model.predict(X)
df["score"] = df["predicted_ctr"] - df["actual_ctr"]

def archetype(pos):
    if pos <= 6: return "high_rank_ignored"
    elif pos <= 10: return "mid_rank_ignored"
    else: return "low_rank_ignored"

def action_for(arch):
    return {
        "high_rank_ignored": "review_snippet_title",
        "mid_rank_ignored": "review_snippet_title + content_relevance_check",
        "low_rank_ignored": "improve_ranking_signals",
    }[arch]

df["archetype"] = df["avg_position"].apply(archetype)
df["action"] = df["archetype"].apply(action_for)
df["reason_code"] = df["score"].apply(
    lambda s: "CTR_BELOW_MODEL_EXPECTED" if s > SCORE_THRESHOLD else "PERFORMING_AS_EXPECTED"
)

ranked_queue = df[df["reason_code"] == "CTR_BELOW_MODEL_EXPECTED"].sort_values(
    ["score", "impressions"], ascending=[False, False]
)
print(f"{len(ranked_queue)} of {len(df)} items flagged for action ({len(ranked_queue)/len(df)*100:.1f}%)")
ranked_queue[["content_hash_id", "archetype", "action", "avg_position", "impressions",
              "actual_ctr", "predicted_ctr", "score", "reason_code"]].head(15)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Writes the ranked queue to CSV, the full results table (including permutation importance) to JSON, and the feature-importance chart to PNG — these are the exact numbers the deployed paper's tables and charts show, so the page never has to be updated by hand.

In [ ]:
!pip install matplotlib --quiet

import os
import json
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked_queue.to_csv("work/outputs/content_action_playbook.csv", index=False)

metrics = {
    **results_table,
    "queue_size": len(ranked_queue),
    "total_eligible_items": len(df),
    "queue_fraction_pct": round(len(ranked_queue) / len(df) * 100, 1),
    "eligibility_threshold_impressions": ELIGIBILITY_MIN_IMPRESSIONS,
    "score_threshold": SCORE_THRESHOLD,
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

importance_series = pd.Series(importance_dict).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importance_series.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Permutation importance (mean)")
ax.set_title("Feature importance — validated Random Forest")
plt.tight_layout()
plt.savefig("work/figures/feature_importance.png", dpi=150)
plt.show()

## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.